In [ ]:
import os
import pandas as pd
import commons as c

# Merge and save DFs for equiv, normal and balanced

In [ ]:
def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path)
            # df = df.loc[~df['Name'].str.contains("Replace", na=False)]
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df

# Function to split the name column and create new columns
def split_name_column(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    
    if len(parts) > 7: 
        parameters = parts[7].strip('[]') 
    else: 
        parameters = None

    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap", "ch"]
    if gate in single_qubit_gates:
        return 'Single_qubit'
    elif gate in multi_qubit_gates:
        return 'Multi_qubit'
    else:
        return 'Gate_not_supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'beginning'
    elif percentage <= 40:
        return 'pre_middle'
    elif percentage <= 60:
        return 'middle'
    elif percentage <= 80:
        return 'post_middle'
    else:
        return 'end'

In [ ]:
def process_characteristics(file_path):
    """Processes the characteristics Excel file into a DataFrame."""
    df_charac = pd.read_excel(file_path, usecols=[0, 2, 3, 5, 6, 7])
    df_charac['algo'] = df_charac.iloc[:, 0].str.split('_').str[0]
    df_charac['qubits'] = df_charac['qubits'].astype(str)
    return df_charac.drop(columns=[df_charac.columns[0]])

In [ ]:
def get_dataframe(model, mutant, threshold, df_char):
    """Generates a processed DataFrame for a given noise model, mutant type, and threshold."""

    # Get all the results in a df
    folder_path = f'./results_{model}/results_{mutant}_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Categorize input type
    df['Input_type'] = df['Input'].str.split('_').str[0]

    # Split 'Name' column into multiple columns
    df[['Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

    # Categorize gate type
    df['Gate_type'] = df['Gate'].apply(get_gate_type)

    # Calculate position percentage and categorize
    df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
    df['position_percentage'] = (df['Position'] / df['max_position']) * 100
    df['Relative_position'] = df['position_percentage'].apply(categorize_position)

    # Drop intermediate columns
    df = df.drop(columns=['max_position', 'position_percentage', 'Name'])

    # Merge with characteristics DataFrame
    merged_df = pd.merge(df_char, df, left_on=['qubits', 'algo'], right_on=['Qubits_number', 'Algorithm'], how='right')
    merged_df = merged_df.drop(columns=['qubits', 'algo'])

    # Map output types
    merged_df['Output_type'] = merged_df['Algorithm'].map(c.output_type)

    # Save DataFrame to CSV
    csv_path = f'results/dataframes/{model}_{mutant}_{threshold}.csv'
    merged_df.to_csv(csv_path, mode='w', header=True, index=False)

    return merged_df

In [ ]:
def get_balanced_df(csv_path, df_equiv, df_normal):
    """Creates a balanced DataFrame by sampling equal rows from two DataFrames."""
    
    n_rows = min(len(df_equiv), len(df_normal))
    df_equiv_sampled = df_equiv.sample(n=n_rows, random_state=42)
    df_normal_sampled = df_normal.sample(n=n_rows, random_state=42)

    # Combine and shuffle
    balanced_df = pd.concat([df_equiv_sampled, df_normal_sampled]).sample(frac=1, random_state=1).reset_index(drop=True)
    balanced_df['Output_type'] = balanced_df['Algorithm'].map(c.output_type)

    # Save to CSV
    balanced_df.to_csv(csv_path, mode='w', header=True, index=False)

    return balanced_df

In [ ]:
def process_metrics(complete_df):
    """Processes metrics and saves results to CSV."""

    selected_columns = complete_df[[
        'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Input', 'Input_type',
        'Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits',
        'Gate_type', 'Relative_position', 'Output_type', 'hardware', 'threshold'
    ]]

    new_rows = []

    for metric, metric_name in c.metrics.items():
        metric_df = selected_columns.copy()
        metric_df['metric'] = metric
        metric_df['true_label'] = complete_df[f'Killed_I{metric}']
        metric_df['predicted_label'] = complete_df[f'Killed_N{metric}']
        metric_df['ideal_distance'] = complete_df[f'Ideal_{metric_name}']
        metric_df['noisy_distance'] = complete_df[f'Noisy_{metric_name}']
        metric_df['correctness'] = metric_df['true_label'] == metric_df['predicted_label']
        new_rows.append(metric_df)

    final_df = pd.concat(new_rows, ignore_index=True)
    final_df = final_df.astype(c.type_dict)
    return final_df

In [ ]:
xlsx_path = 'data/origin_qc/programs_characteristics.xlsx'
df_charac = process_characteristics(xlsx_path)
os.makedirs('results/dataframes/', exist_ok=True)

mutant_typed_df = {"equiv": [], "normal": [], "balanced": []}

for threshold in c.thresholds:
    for hw in c.hardware:
        # Load 'equiv' and 'normal' DataFrames once
        df_equiv = get_dataframe(hw, "equiv", threshold, df_charac)
        df_normal = get_dataframe(hw, "normal", threshold, df_charac)
        csv_balanced_path = f'results/dataframes/{hw}_balanced_{threshold}.csv'
        df_balanced = get_balanced_df(csv_balanced_path, df_equiv, df_normal)
        
        mutant_temp_df = {"equiv": df_equiv, "normal": df_normal, "balanced": df_balanced}
        
        for mutant_type, df in mutant_temp_df.items():
            df['hardware'] = hw
            df['threshold'] = threshold
            mutant_typed_df[mutant_type].append(df)

for mutant_type, df in mutant_typed_df.items():
    complete_df = pd.concat(df, ignore_index=True)
    new_df = process_metrics(complete_df)
    
    output_folder = 'results/dataframes/'
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f'results_{mutant_type}.csv')
    new_df.to_csv(output_path, index=False)
